# 수익률 방향 비대칭 · 시장 레짐 · 공매도 제약 — 한국/미국 실증 결과 보고서

- 작성일: 2026-09-28
- 실행 범위: 0~8단계 전체 (실제 yfinance · KRX Open API 데이터)
- 동결 기록: `S1` 2026-09-27 09:32 / `S2S3` 2026-09-28 15:18 (`config/frozen/`)
- 모든 수치의 원본: `results/step1` ~ `results/step7`, `logs/paper_s1.csv`

---

## 0. 요약

| 가설 / 전략 | 결론 | 핵심 근거 |
|---|---|---|
| **H1** 하락이 미래 변동성을 더 키움 | **지지 (한미 모두)** | LHAR이 HAR 대비 표본외 QLIKE 3.6%(미국)·3.8%(한국) 개선, DM t = 3.6 / 3.9 |
| H1-b 혼란 레짐에서 더 커짐 | **지지 (한미 모두)** | 레짐 LHAR의 `L×D` 계수 t = 2.83(미국), 2.62(한국). RLHAR 개선폭 5.0% / 4.4% |
| **H2** RSJ·RSkew·MAX·IVOL의 횡단면 예측력 | **기각 (한미 모두)** | 사전 기준 통과 시그널 0개. 미국 RSJ의 약한 예측력은 반전(REV)으로 설명됨 |
| **H3** H2가 레짐에 따라 다름 | **약한 증거만** | 한국 RSJ IC가 혼란 레짐에서 −0.080 (t = −1.93, 혼란 9개월). 미국은 유의하지 않음 |
| **H4** 공매도 금지 기간 한국만 효과 변화 | **기각** | 국가 간 DiD θ₃ 모두 유의하지 않음 (|t| ≤ 1.35), 플라시보 p = 0.29 ~ 0.85 |
| **H5** 한국과 미국의 효과 차이 | **대체로 차이 없음** | 같은 달 IC 차이 모두 유의하지 않음. 레짐 상호작용에서 RSJ만 t = −2.16 (다중검정 고려 시 약함) |
| **S1** 레짐/변동성 기반 주식 비중 | **개발 구간 우수 → 테스트에서 관문 탈락** | 테스트 샤프: 미국 0.68 vs 200일선 0.80, 한국 0.61 vs 200일선 0.74 |
| **S2 / S3** 종목 선택 / 결합 | **생략 (결정 관문)** | H2 기각으로 선택된 시그널 없음 → 빈 규칙으로 동결 |

한 줄로 정리하면: **하락 변동성의 "시계열" 효과(H1)는 두 시장 모두에서 견고하지만, "횡단면" 수익률 예측(H2~H5)은 월간 일봉 기반 지표로는 나타나지 않았다.** 전략 측면에서 S1은 개발 구간에서 낙폭을 크게 줄였지만 2021년 이후 테스트 구간에서는 단순한 200일선·변동성 타기팅을 이기지 못했다.

---

## 1. 데이터와 설계

### 1.1 기간

| 구간 | 기간 | 쓰임 |
|---|---|---|
| 데이터 시작 | 2000-01 | 레짐 모형 학습 (닷컴 붕괴·2008년 포함) |
| 지수 개발 구간 (1~2단계) | 2005-01 ~ 2020-12 | HAR/LHAR, 레짐, S1 설계 |
| 횡단면 개발 구간 (4~6단계) | **2010-01 ~ 2020-12** (`CS_START`) | H2·H3·H5, H4 플라시보, S2 설계 |
| 테스트 구간 | 2021-01 ~ 2026-09 | 규칙 동결 후 1회만 평가 |
| 한국 공매도 전면 금지 | 2023-11 ~ 2025-03 | H4 |

횡단면 시작을 2010년으로 옮긴 결정은 **4단계 결과를 보기 전(2026-09-27)** 에 했고 README "설정 변경 기록"에 남겼다 (한국 Open API 제공 기간, 미국 생존편향).

### 1.2 종목 데이터 (3단계)

| | 미국 | 한국 |
|---|---|---|
| 출처 | yfinance (종목별) | KRX Open API 유가증권 일별매매정보 (날짜별 전종목, 4,366 거래일) |
| 유니버스 | S&P 500 과거 구성종목 (`sp500_history.csv`) | 월말 시가총액 상위 200 보통주 (KOSPI200 대리) |
| 구성종목 합집합 | 1,008개 | 1,251개 (KOSPI 전종목 중 보통주 기준 상위 200에 한 번이라도 든 종목 포함) |
| 데이터 누락 비율 (월평균 / 최대) | **22.1% / 40.6%** | **0.0% / 0.0%** |
| 패널 | 112,809행, 599종목 (2000-01 ~ 2026-09) | 39,950행, 464종목 (2010-01 ~ 2026-09) |
| 개발 구간 월평균 종목 수 | 370 | 199 |
| SIZE 변수 | 거래대금(대리) | 시가총액 |

- **미국은 생존편향이 크다.** 구성종목 1,008개 중 340개를 yfinance가 제공하지 않았고 대부분 상장폐지 종목이다. 누락은 과거로 갈수록 크다.
- **한국은 생존편향이 없다.** 날짜별 전종목 자료라 이후 상장폐지된 종목도 그 시점 유니버스에 들어간다.
- **한국 공매도 잔고는 받지 못했다.** pykrx가 KRX 로그인을 요구해 `kr_short_ratio_preban.csv` 생성에 실패했다 → H4의 **한국 내 DiD는 생략**됐고 국가 간 DiD만 수행했다.

### 1.3 지수 데이터 품질 (1단계)

`^GSPC`, `SPY`, `^KS11`은 결측·가격 오류 없음. `069500.KS`(KODEX 200)는 2007-01 시작이고 최대 800일 공백이 있어, 한국 S1 개발 구간 평가는 약 11.5년이다 (미국은 16년).

---

## 2. H1 — 하락은 미래 변동성을 더 키우는가 (1단계)

### 2.1 HAR vs LHAR (표본외, 워크포워드, 로그 모형)

| 시장 | 모형 | 표본외 일수 | QLIKE | 개선 (vs HAR) | DM t |
|---|---|---|---|---|---|
| 미국 | HAR | 4,027 | 0.4015 | — | — |
| 미국 | **LHAR** | 4,027 | **0.3872** | **+3.56%** | **3.64** |
| 한국 | HAR | 3,952 | 0.3375 | — | — |
| 한국 | **LHAR** | 3,952 | **0.3246** | **+3.81%** | **3.87** |

수준(선형) 모형은 음수·극소 예측 때문에 QLIKE가 크게 나빠져(LHAR_level −157% ~ −599%) 비교용으로만 둔다 (README 한계 8).

### 2.2 LHAR 계수 (개발 구간, NW t)

| 계수 | 미국 | 한국 |
|---|---|---|
| L (전일 하락) | 0.200 (2.64) | 0.143 (1.14) |
| L_w (주간 하락) | **2.141 (4.45)** | **1.403 (2.37)** |
| L_m (월간 하락) | **3.066 (2.97)** | **2.238 (2.52)** |

하락 수익률 항이 주·월 단위에서 두 시장 모두 유의하게 양(+)이다 → **레버리지 효과 확인**.

### 2.3 레짐 LHAR (2단계) — 혼란 레짐에서 더 커지는가

| 계수 | 미국 | 한국 |
|---|---|---|
| D (혼란 더미) | −0.342 (−1.33) | −0.484 (−1.33) |
| lRV × D | **0.344 (2.22)** | **1.579 (2.82)** |
| **L × D** | **0.503 (2.83)** | **0.663 (2.62)** |

| 표본외 QLIKE 개선 (vs HAR) | 미국 | 한국 |
|---|---|---|
| LHAR | +3.64% (DM t 3.25) | +3.37% (DM t 2.98) |
| **RLHAR (레짐 LHAR)** | **+4.99% (DM t 3.21)** | **+4.40% (DM t 2.78)** |

혼란 레짐에서 하락의 변동성 영향이 추가로 커지고(L×D > 0), 이를 반영한 RLHAR이 표본외에서도 LHAR보다 낫다. **H1은 두 부분 모두 한미에서 지지된다.**

---

## 3. 레짐 모형 진단 (2단계)

주 모형: Jump Model(JM), 변동성 피처, K = 2, λ는 개발 구간 S1 샤프로 선택 (후보 5개).

| 항목 | 미국 | 한국 |
|---|---|---|
| 선택 λ | 5 | 5 |
| 혼란 레짐 비중 | 25.5% | 12.2% |
| 레짐 연 변동성 (평온 / 혼란) | 11.9% / 33.5% | 15.2% / 39.5% |
| 평균 지속기간 (평온 / 혼란, 일) | 188 / 69 | 315 / 48 |
| 연간 전환 횟수 | 1.88 | 1.27 |
| online–offline 일치율 | 96.7% | 94.3% |
| HMM과의 일치율 | 88.0% | 91.3% |
| 탐지 지연 중앙값 | 1일 | 3일 |

사건별 혼란 레짐 비중:

| 사건 | 미국 | 한국 |
|---|---|---|
| GFC (2008.9~2009.3) | 100% | 100% |
| 유럽 위기 (2011 하반기) | 97% | 77% |
| 중국 쇼크 (2015.8~2016.2) | 48% | **0%** |
| 2018 4분기 | 71% | **0%** |
| COVID (2020.2~4) | 92% | 84% |

레짐은 지속적이고(평균 수십~수백 일) 실시간 필터링과 사후 추정이 잘 일치한다. 다만 한국은 2015·2018 조정을 혼란으로 분류하지 않았다 — 한국 지수 자체의 변동성 상승이 상대적으로 작았던 결과로, 한국 레짐이 "큰 위기만 잡는" 성격이라는 점을 해석 시 고려해야 한다.

---

## 4. S1 — 레짐/변동성 기반 주식 비중

동결 규칙 (`config/frozen/S1.json`): 두 시장 모두 **규칙 A**(레짐 스위치, 혼란 레짐 비중 0), JM λ = 5, 버퍼 0.

### 4.1 개발 구간 (미국 2005~2020, 한국 약 11.5년)

| 전략 | 미국 샤프 | 미국 MDD | 한국 샤프 | 한국 MDD |
|---|---|---|---|---|
| **S1_A (동결)** | **0.71** | **−15.1%** | 0.54 | −25.2% |
| S1_B (LHAR 변동성 타기팅) | 0.63 | −37.9% | 0.50 | −27.2% |
| S1_C (결합) | 0.70 | −13.1% | 0.46 | −25.8% |
| 바이앤홀드 | 0.50 | −55.2% | **0.57** | −38.1% |
| 200일선 | 0.49 | −19.1% | 0.20 | −48.2% |
| 단순 변동성 타기팅 | 0.63 | −30.7% | 0.48 | −26.4% |

개발 구간에서 미국 S1은 샤프와 낙폭 모두 벤치마크보다 확실히 좋았다. 한국은 낙폭은 줄였지만 샤프는 바이앤홀드와 비슷하거나 약간 낮았다.

### 4.2 테스트 구간 (2021-01 ~ 2026-09, 동결 후 1회)

| 전략 | 미국 연수익 | 미국 샤프 | 미국 MDD | 한국 연수익 | 한국 샤프 | 한국 MDD |
|---|---|---|---|---|---|---|
| **S1_A (동결)** | 10.8% | **0.68** | −18.0% | 9.3% | **0.61** | −38.4% |
| 바이앤홀드 | 15.1% | 0.73 | −24.5% | 23.7% | 0.85 | −40.8% |
| 200일선 | 12.5% | **0.80** | −18.1% | 17.1% | **0.74** | −45.6% |
| 단순 변동성 타기팅 | 13.5% | 0.80 | −15.7% | 12.5% | 0.83 | −31.0% |
| (참고) S1_B | 13.8% | 0.79 | −19.9% | 16.1% | 0.90 | −35.8% |

**결정 관문 (2단계 후): "S1이 테스트에서 200일선 규칙도 못 이김" → 두 시장 모두 해당.**
README 관문 규칙에 따르면 동결된 S1_A를 그대로 페이퍼 트레이딩하지 않는다.

해석:
- 샤프 표준오차가 약 0.46 ~ 0.51이라, 테스트 구간 샤프 차이(0.1 ~ 0.2)는 통계적으로 구분되지 않는다. "S1이 나쁘다"기보다 "S1의 우위가 테스트에서 재현되지 않았다"가 정확한 표현이다.
- 테스트 구간에서 S1_A의 연수익이 바이앤홀드보다 4~14%p 낮다. 혼란 레짐에서 비중을 0으로 빼는 규칙 A가 급락 뒤 반등 구간을 놓친 비용으로 추정된다. 한국 MDD(−38%)가 바이앤홀드(−41%)와 거의 같은 것은, 완만하게 진행된 하락에서는 변동성 기반 레짐 스위치가 늦게 반응할 수 있음을 시사한다. (둘 다 일별 비중 경로를 따로 분해하지 않은 추정이다.)
- 사후적으로 S1_B가 테스트에서 더 좋았지만, 이것을 보고 규칙을 바꾸면 테스트 구간 과적합이다. 바꾸려면 새 버전으로 동결하고 **페이퍼 트레이딩으로만** 검증해야 한다.

---

## 5. H2 — 횡단면 수익률 예측 (4단계, 2010~2020, 131개월)

### 5.1 Rank IC 요약

| 시그널 | 미국 평균 IC | 미국 t(NW) | 미국 전/후반 | 한국 평균 IC | 한국 t(NW) | 한국 전/후반 |
|---|---|---|---|---|---|---|
| RSJ | −0.018 | −1.57 | −0.010 / −0.026 | +0.001 | 0.13 | **+0.023 / −0.020** |
| RSkew | −0.014 | −1.30 | −0.008 / −0.020 | +0.003 | 0.27 | +0.023 / −0.018 |
| MAX | −0.001 | −0.04 | +0.001 / −0.002 | −0.004 | −0.40 | +0.017 / −0.025 |
| IVOL | +0.005 | 0.39 | +0.007 / +0.004 | −0.011 | −0.97 | +0.017 / −0.039 |
| REV | −0.020 | −1.52 | −0.012 / −0.028 | −0.003 | −0.28 | +0.020 / −0.027 |

### 5.2 중립화 IC (통제변수 제거 후) — S2 선택 기준의 핵심

| 시그널 | 미국 원 IC → 중립화 IC (t) | 한국 원 IC → 중립화 IC (t) |
|---|---|---|
| RSJ | −0.018 → **+0.004 (0.43)** | +0.001 → +0.009 (1.19) |
| RSkew | −0.014 → +0.004 (0.46) | +0.003 → +0.007 (0.84) |
| MAX | −0.001 → −0.002 (−0.14) | −0.004 → −0.005 (−0.60) |
| IVOL | +0.005 → −0.001 (−0.08) | −0.011 → −0.010 (−1.17) |
| REV | −0.020 → −0.020 (−1.55) | −0.003 → −0.006 (−0.54) |

### 5.3 RSJ는 반전 효과의 재포장인가

| 증거 | 미국 | 한국 |
|---|---|---|
| RSJ–REV 순위상관 | **0.85** | **0.87** |
| RSJ–RSkew 순위상관 | 0.96 | 0.96 |
| FM: RSJ 단독 | −0.0051 (t −2.08) | +0.0039 (1.08) |
| FM: RSJ + REV | +0.0026 (0.56) | +0.0010 (0.19) |
| FM: 전체 통제 (6) | −0.0001 (−0.02) | −0.0070 (−0.38) |
| 이중정렬 RSJ \| REV 롱숏 | −0.0008 (t −0.47) | −0.0001 (t −0.03) |
| 이중정렬 RSJ \| IVOL 롱숏 | +0.0034 (t 1.81) | +0.0028 (t 1.17) |

미국에서 RSJ 단독 FM 계수는 유의(t = −2.08)하지만 **REV를 넣는 순간 사라지고**, REV 안에서 RSJ로 다시 정렬하면 롱숏 수익이 0이다. 일봉으로 만든 월간 RSJ는 그 달 수익률과 기계적으로 얽혀 있어(README 한계 4), 월간 해상도에서는 반전과 구분되는 정보가 없다.

### 5.4 분위 포트폴리오 (동일가중, 월 롱숏)

| 시그널 | 미국 LS (t) | 미국 단조성 ρ | 한국 LS (t) | 한국 단조성 ρ |
|---|---|---|---|---|
| RSJ | 0.35% (1.69) | −0.9 | 0.52% (1.65) | 0.5 |
| RSkew | 0.30% (1.61) | −0.9 | 0.35% (1.17) | 0.2 |
| MAX | −0.15% (−0.46) | 0.1 | −0.58% (−1.64) | 0.7 |
| IVOL | 0.18% (0.60) | 0.9 | −0.55% (−1.55) | 0.9 |
| REV | 0.40% (1.60) | −1.0 | −0.44% (−1.22) | 0.1 |

시가총액 가중(SW)으로 바꾸면 모든 롱숏이 0 근처(|t| < 0.8)로 줄어든다 → 약한 효과마저 소형주에 몰려 있다.

### 5.5 예측 지평 (IC decay)

미국 RSJ·REV의 음(−)의 IC는 1~2개월에 가장 크고 3개월 이후 사라진다 (단기 반전의 전형적 모양). 한국 IVOL·MAX는 지평이 길수록 IC가 더 음(−)으로 커지지만(6개월 IVOL −0.032, t −1.68) 유의 수준에는 못 미친다.

### 5.6 S2 선택 결과 (사전 확정 기준 `S2_CRITERIA`)

기준: 중립화 IC |t| ≥ 2, 전·후반 부호 일치, 분위 단조성 |ρ| ≥ 0.8, 롱 다리 − 벤치마크 > 0.

| 시그널 | 미국 탈락 사유 | 한국 탈락 사유 |
|---|---|---|
| RSJ | 중립화 t 부족 | 중립화 t 부족, 전/후반 부호 불일치, 단조성 부족 |
| RSkew | 중립화 t 부족 | 중립화 t 부족, 전/후반 부호 불일치, 단조성 부족 |
| MAX | 중립화 t 부족, 전/후반 부호 불일치, 단조성 부족 | 중립화 t 부족, 전/후반 부호 불일치, 단조성 부족, 롱 다리 음수 |
| IVOL | 중립화 t 부족 | 중립화 t 부족, 전/후반 부호 불일치, 롱 다리 음수 |
| REV | 중립화 t 부족 | 중립화 t 부족, 전/후반 부호 불일치, 단조성 부족, 롱 다리 음수 |

**통과 0개 (두 시장 모두) → H2 기각.** 기준은 미국 결과를 본 뒤에도 바꾸지 않았다. 한국은 모든 시그널에서 개발 구간 전반(2010~2015)과 후반(2015~2020) IC 부호가 뒤집혀, 안정적인 횡단면 효과가 없다는 점이 특히 뚜렷하다.

---

## 6. H3 — 레짐 조건부 횡단면 효과 (4단계 T9)

| 항목 | 미국 평온 평균 | 미국 혼란−평온 (t) | 한국 평온 평균 | 한국 혼란−평온 (t) |
|---|---|---|---|---|
| IC_RSJ | −0.011 | −0.034 (−1.20) | +0.007 | **−0.080 (−1.93)** |
| IC_RSkew | −0.008 | −0.026 (−0.99) | +0.007 | −0.061 (−1.57) |
| IC_MAX | −0.002 | +0.008 (0.20) | −0.003 | −0.014 (−0.41) |
| IC_IVOL | +0.001 | +0.018 (0.53) | −0.012 | +0.016 (0.52) |
| IC_REV | −0.014 | −0.030 (−0.90) | +0.003 | −0.099 (−1.91) |
| FM_RSJ (전체 통제) | −0.004 | +0.017 (0.68) | −0.002 | −0.068 (−1.31) |
| **횡단면 분산 σ_cs** | 0.061 | **+0.018 (3.60)** | 0.092 | **+0.032 (2.58)** |
| 월 수 (평온 / 혼란) | 103 / 28 | | 122 / 9 | |

- 두 시장 모두 **혼란 레짐에서 종목 간 수익률 분산이 유의하게 커진다** (확실한 결과).
- 시그널의 예측력이 레짐에 따라 달라진다는 증거는 약하다. 한국 RSJ·REV IC가 혼란 레짐에서 더 음(−)으로 가지만 t ≈ −1.9이고, **혼란 월이 9개월뿐**이라 신뢰하기 어렵다. 또 RSJ와 REV가 같이 움직이므로 이것도 반전 효과의 레짐 의존성일 가능성이 크다.
- 미국 혼란 에피소드별 RSJ IC는 9개 중 7개가 음(−)이지만 크기가 제각각이다 (−0.38 ~ +0.08).

**H3: 약한 증거만 있음 (유의하지 않음).**

---

## 7. H5 — 한국과 미국의 차이 (5단계 pre, 2010~2020)

### 7.1 같은 달 IC 차이 (한국 − 미국)

| 시그널 | 평균 차이 | t(NW) |
|---|---|---|
| RSJ | +0.019 | 1.41 |
| RSkew | +0.016 | 1.24 |
| REV | +0.017 | 1.10 |
| MAX | −0.004 | −0.18 |
| IVOL | −0.016 | −0.89 |

모두 유의하지 않다. 방향만 보면 미국은 RSJ·REV가 음(−)의 IC(반전), 한국은 0 근처라 한국의 단기 반전이 약하다.

### 7.2 한국 레짐별 차이

| 시그널 | 평온기 차이 | 혼란기 추가 차이 (t) |
|---|---|---|
| RSJ | +0.025 | **−0.084 (−2.16)** |
| REV | +0.022 | −0.077 (−1.99) |
| RSkew | +0.021 | −0.066 (−1.57) |
| IVOL | −0.022 | +0.082 (1.67) |
| MAX | −0.006 | +0.031 (0.54) |

한국 혼란기에는 한미 차이가 사라지거나 뒤집힌다 (평온기 한국의 반전이 약하다가, 위기 때 미국처럼 반전이 나타남). RSJ만 |t| > 2지만 5개 시그널을 동시에 검정했으므로(본페로니 기준 |t| ≈ 2.6) **확정적 결과로 보기 어렵다.**

**H5: 대체로 차이 없음. 레짐 상호작용은 추가 검증 필요.**

---

## 8. H4 — 한국 공매도 금지 효과 (5단계 post)

### 8.1 국가 간 DiD (한국 FM 계수 − 미국 FM 계수, 레짐 통제)

| 시그널 | θ₃ (금지 기간) | t(NW) | Post (해제 후) | 플라시보 p | 한국 고유 사건 월 제외 θ₃ (t) |
|---|---|---|---|---|---|
| RSJ | −0.024 | −1.21 | +0.009 | 0.33 | −0.026 (−1.12) |
| RSkew | −0.016 | −1.35 | −0.003 | 0.29 | −0.018 (−1.33) |
| IVOL | +0.008 | 0.66 | −0.007 | 0.62 | +0.003 (0.23) |
| MAX | −0.003 | −0.23 | −0.005 | 0.85 | −0.006 (−0.49) |

플라시보: 개발 구간(2010~2020) 안의 같은 길이(17개월) 가짜 금지 기간 115개로 만든 분포. 실제 θ₃가 이 분포의 꼬리에 있지 않다 (p = 0.29 ~ 0.85). 2024-02(밸류업 발표), 2024-12(계엄 사태) 제외도 결과를 바꾸지 않는다.

### 8.2 한국 내 DiD (공매도 잔고 High vs Low)

**생략.** 금지 직전 공매도 잔고 데이터(pykrx)를 KRX 로그인 문제로 받지 못했다. 5단계 pre의 리허설(가짜 기간·가짜 분류)로 코드가 동작하는 것만 확인했다.

**H4: 기각.** 공매도 금지 기간에 한국에서만 시그널의 횡단면 효과가 달라졌다는 증거는 없다. 다만 애초에 H2의 효과 자체가 0에 가까워 "바뀔 효과가 없었다"는 점도 함께 기억해야 한다.

---

## 9. S2 · S3 (6단계)

- 4단계에서 선택된 시그널이 없어 **결정 관문 "4단계 후 → S2 생략(빈 규칙으로 동결), 7단계 비중 확대"** 가 적용됐다.
- `S2S3`는 2026-09-28 15:18:51에 빈 규칙 `{}`으로 동결됐다 (sha256 `44136fa355b3…`). 이 동결은 테스트 구간 데이터를 쓰는 5단계 post·7단계를 열기 위한 관문 통과용이다.
- 6단계 `--phase test`는 평가할 규칙이 없어 산출물이 없다. S3가 "ETF+S1"을 이기는지에 대한 관문은 판단 대상이 아니다.

---

## 10. 장중 데이터 검증 (7단계, 예비 결과)

장중 데이터는 2026-09-27 한 번만 수집했으므로 5분봉은 약 60거래일, 60분봉은 약 150주다.

### 10.1 5분봉 실현변동성 vs Parkinson (일봉 레인지)

| | 종목 수 | 상관 중앙값 | Parkinson / RV5m 중앙값 | 일간 RSJ 중앙값 |
|---|---|---|---|---|
| 미국 | 63 | 0.70 | 1.21 | −0.007 |
| 한국 | 52 | 0.75 | 1.14 | +0.022 |

일봉 레인지로 만든 변동성은 5분봉 실현변동성과 상관 0.7 수준이고 평균적으로 14~21% 크게 나온다 (호가 튐·장 시작 갭의 영향). 1~2단계의 결론(H1)을 바꿀 수준의 괴리는 아니다. 5분봉 HAR 계열 비교는 표본(60일)이 최소 요구치(120일)에 못 미쳐 실행되지 않았다.

### 10.2 60분봉 주간 RSJ의 다음 주 수익률 예측

| | 주 수 | 종목 수 | 평균 주간 IC | t(NW) |
|---|---|---|---|---|
| 미국 | 151 | 62 | −0.019 | −0.91 |
| 한국 | 143 | 51 | −0.017 | −1.05 |

장중 해상도로 만든 RSJ도 다음 주 수익률을 유의하게 예측하지 못했다. 다만 종목 수가 50~60개(수집 대상 상위 50종목 + 기준 종목)로 작아 검정력이 낮다. 결정 관문에 따라 7단계 비중을 늘리려면 **장중 수집을 매주 계속하고 수집 종목 수를 늘려야** 한다.

---

## 11. 페이퍼 트레이딩 (8단계, 시작 상태)

| 시장 | 데이터 날짜 | 레짐 | 예측 분산 | 다음 거래일 목표 비중 |
|---|---|---|---|---|
| 미국 | 2026-09-25 | 0 (평온) | 0.243 | 1.00 |
| 한국 | 2026-09-28 | **1 (혼란)** | 1.918 | **0.00** |

- 한국은 최근 30거래일 일간 변동성이 2.5%(8/19 −6.0% 포함)로 커져 혼란 레짐이다. 미국은 0.6%로 평온하다.
- **주의 1:** 한국 기록은 15:21, 장 마감(15:30) 전에 만들었다. 16시 이후 `python scripts/step8_paper_trading.py --mode s1-update --market KR`을 다시 실행하면 같은 날짜 기록이 확정값으로 덮어써진다.
- **주의 2:** 4.2절의 결정 관문에 따라 동결된 S1_A를 "실전 후보"로 페이퍼 트레이딩하는 것은 권하지 않는다. 계속 기록한다면 실전 전환 목적이 아니라 **레짐 신호 모니터링** 용도로만 쓰는 것이 원칙에 맞다.
- S2 페이퍼는 동결된 S2 규칙이 없어 해당 없음.

---

## 12. 결정 관문 정리

| 시점 | 조건 | 결과 | 결정 |
|---|---|---|---|
| 2단계 후 | S1이 테스트에서 200일선도 못 이김 | **해당 (한미 모두)** | S1_A를 그대로 페이퍼 트레이딩하지 않음 |
| 4단계 후 | 기준 통과 시그널 없음 | **해당 (한미 모두)** | S2 생략(빈 규칙 동결), 7단계 비중 확대 |
| 5단계 후 | 플라시보에서도 비슷한 효과 | 해당 (θ₃가 플라시보 분포 안) | "공매도 효과"로 주장하지 않음 |
| 6단계 후 | S3가 ETF+S1을 못 이김 | 판단 대상 없음 (S3 미구성) | 8단계는 S1만 (그마저 관문 1에 걸림) |

---

## 13. 한계

1. **미국 생존편향**: 구성종목 월평균 22%(최대 41%) 누락. 누락 종목은 대부분 상장폐지 종목이라, 미국 횡단면 결과는 "살아남은 대형주" 표본이다. 횡단면 시작을 2010년으로 옮겨 가장 심한 구간은 피했다.
2. **미국 SIZE = 거래대금**: 과거 시가총액이 없어 규모와 유동성이 섞인 대리변수를 썼다.
3. **월간 RSJ와 반전의 기계적 연관**: 순위상관 0.85~0.87. 월간 일봉 해상도로는 두 효과를 분리할 수 없고, 이번 결과의 핵심 제약이다.
4. **한국 유니버스 = 시가총액 상위 200 대리**: 실제 KOSPI200과 일부 다르다. 배당 미포함 가격수익률이다.
5. **한국 혼란 레짐 표본이 작다**: 횡단면 개발 구간 131개월 중 9개월. H3·H5의 한국 레짐 결과는 검정력이 낮다.
6. **H4 한국 내 DiD 미실시**: 공매도 잔고 데이터 미수집.
7. **비용·세금은 가정치** (`config.MARKETS`). 한국 증권거래세는 연도별 실제 세율로 바꾸지 않았다.
8. **장중 검증은 예비**: 수집 1회, 60거래일(5분봉)·약 150주(60분봉), 시장별 50~60종목.
9. **다중검정**: 시그널 5개 × 시장 2개 × 여러 표를 동시에 봤다. 경계선 결과(|t| ≈ 2)는 우연일 수 있다.

---

## 14. 다음 단계 제안

1. **장중 데이터 축적**: `step0_collect_intraday.py`를 매주 자동 실행하고 `INTRADAY_TOP_N`을 늘려, 주간 RSJ의 횡단면 검정력을 확보한다. 이번 결과에서 RSJ가 새로운 정보를 가질 수 있는 유일한 경로는 장중 해상도다.
2. **공매도 잔고 확보**: KRX 로그인을 설정해 `step3 --market KR --short`를 다시 실행하고, H4 한국 내 DiD를 수행한다. 이미 S2S3가 동결됐으므로 이 분석은 5단계 post만 다시 돌리면 된다.
3. **S1 재설계는 새 버전으로**: 테스트 결과를 보고 S1을 고치고 싶다면 (예: 혼란 레짐 비중 0 → 부분 비중, 또는 규칙 B) 새 이름으로 동결하고 **앞으로의 페이퍼 기록으로만** 평가한다. 기존 테스트 구간으로 다시 평가하면 안 된다.
4. **보고서 핵심 주장**: "하락 비대칭은 변동성 예측(H1)에서는 한미 공통으로 강하고 레짐 의존적이나, 월간 횡단면 수익률 예측에서는 반전 효과와 구분되지 않으며, 한국 공매도 금지 전후로도 변화가 없다."

---

## 부록 A. 이번 실행에서 바꾼 코드

| 파일 | 변경 | 이유 |
|---|---|---|
| `src/config.py`, `scripts/step4~6` | `CS_START = "2010-01-01"` 도입, T3 분할 2015-06 | 4단계 결과를 보기 전 결정 (README 설정 변경 기록) |
| `scripts/_bootstrap.py` | 표준 출력을 UTF-8로 재설정 | Windows cp949 콘솔에서 `—` 출력 시 미국 3단계가 패널 저장 직전 중단됨 |
| `src/krx_api.py` | 빈 응답(JSON 아님)을 일시 오류로 보고 최대 3회 재시도 | KRX가 간헐적으로 빈 본문을 반환해 수집이 중단됨 (2013-11-08, 2026-03-27 발생) |

분석 규칙(`S2_CRITERIA`, 시그널 목록, H4 시그널, 레짐 설정)은 결과를 본 뒤 바꾸지 않았다.

## 부록 B. 테스트 구간 접근 기록

`logs/test_access.log`에 방화벽을 통과한 모든 테스트 구간 접근이 남아 있다: 2단계 final test(2026-09-27), 5단계 post·7단계(2026-09-28, S1·S2S3 동결 후). 6단계 test는 평가할 규칙이 없어 테스트 데이터를 읽지 않았으므로 기록이 없다.
